### Problem 4

In [17]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import studentized_range

df     = pd.read_csv("HW3Problem4.csv")
groups = [df[c].values for c in df.columns]
names  = list(df.columns)
I, J   = 4, 20
n      = I * J
means  = [np.mean(g) for g in groups]
grand  = np.mean(means)

# (i)
print("(i) Parameter Estimates")
print(f"mu = {grand:.5f}")
alphas = [m - grand for m in means]
for i, a in enumerate(alphas):
    print(f"alpha_{i+1} = {a:.5f}")

# (ii)
print("\n(ii) Regression Parameters")
print(f"beta0 = {means[0]:.3f}")
for i in range(1, 4):
    print(f"beta{i} = {means[i]-means[0]:.3f}")

# (iii)
print("\n(iii) ANOVA")
SSTr = J * sum((m - grand)**2 for m in means)
SSE  = sum(sum((y-m)**2 for y in g) for g,m in zip(groups,means))
SST  = SSTr + SSE
MSTr = SSTr / (I-1)
MSE  = SSE  / (n-I)
F    = MSTr / MSE
p    = 1 - stats.f.cdf(F, I-1, n-I)
print(f"SSTr={SSTr:.3f}, SSE={SSE:.3f}, SST={SST:.3f}")
print(f"MSTr={MSTr:.3f}, MSE={MSE:.3f}")
print(f"F({I-1},{n-I}) = {F:.4f}, p-value = {p:.6f}")

# (iv) Tukey
print("\n(iv) Tukey HSD")
sigma = np.sqrt(MSE)
q     = studentized_range.ppf(0.95, I, n-I)
HSD   = q * sigma / np.sqrt(J)
print(f"sigma={sigma:.4f}, q={q:.4f}, HSD={HSD:.4f}")
for i,j in [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]:
    diff = abs(means[i]-means[j])
    sig  = "Reject" if diff > HSD else "Fail to reject"
    print(f"  {names[i]} vs {names[j]}: |diff|={diff:.3f}, {sig}")

# (v) Dunnett
print("\n(v) Dunnett")
CV_D = 2.479 * sigma * np.sqrt(2/J)
print(f"CV_D = {CV_D:.4f}")
for i in range(1, 4):
    diff = abs(means[i]-means[0])
    sig  = "Reject" if diff > CV_D else "Fail to reject"
    print(f"  C vs {names[i]}: |diff|={diff:.3f}, {sig}")

# (vi) Bonferroni
print("\n(vi) Bonferroni")
m_tests = 5
t_crit  = stats.t.ppf(1 - 0.05/(2*m_tests), n-I)
SE      = sigma * np.sqrt(2/J)
SE4     = sigma * np.sqrt((1 + 1/9 + 1/9 + 1/9)/J)
print(f"t_crit = {t_crit:.4f}, SE = {SE:.4f}")
contrasts = [
    (means[0]-means[1], SE,  "mu1-mu2"),
    (means[0]-means[2], SE,  "mu1-mu3"),
    (means[0]-means[3], SE,  "mu1-mu4"),
    (means[0]-(means[1]+means[2]+means[3])/3, SE4, "mu1-avg(F1,F2,F3)"),
    (means[1]-means[2], SE,  "mu2-mu3"),
]
for L, se, label in contrasts:
    t   = L / se
    sig = "Reject" if abs(t) > t_crit else "Fail to reject"
    print(f"  {label}: L={L:.4f}, t={t:.4f}, {sig}")

# (vii) Power
print("\n(vii) Power")
lam   = J * sum(a**2 for a in alphas) / MSE
F_crit = stats.f.ppf(0.95, I-1, n-I)
power  = 1 - stats.ncf.cdf(F_crit, I-1, n-I, lam)
print(f"lambda = {lam:.4f}, F_crit = {F_crit:.4f}, Power = {power:.6f}")

(i) Parameter Estimates
mu = 19.71375
alpha_1 = -3.74875
alpha_2 = 0.58125
alpha_3 = -1.40875
alpha_4 = 4.57625

(ii) Regression Parameters
beta0 = 15.965
beta1 = 4.330
beta2 = 2.340
beta3 = 8.325

(iii) ANOVA
SSTr=746.352, SSE=1824.523, SST=2570.875
MSTr=248.784, MSE=24.007
F(3,76) = 10.3630, p-value = 0.000009

(iv) Tukey HSD
sigma=4.8997, q=3.7149, HSD=4.0700
  C vs F1: |diff|=4.330, Reject
  C vs F2: |diff|=2.340, Fail to reject
  C vs F3: |diff|=8.325, Reject
  F1 vs F2: |diff|=1.990, Fail to reject
  F1 vs F3: |diff|=3.995, Fail to reject
  F2 vs F3: |diff|=5.985, Reject

(v) Dunnett
CV_D = 3.8410
  C vs F1: |diff|=4.330, Reject
  C vs F2: |diff|=2.340, Fail to reject
  C vs F3: |diff|=8.325, Reject

(vi) Bonferroni
t_crit = 2.6421, SE = 1.5494
  mu1-mu2: L=-4.3300, t=-2.7946, Reject
  mu1-mu3: L=-2.3400, t=-1.5102, Fail to reject
  mu1-mu4: L=-8.3250, t=-5.3730, Reject
  mu1-avg(F1,F2,F3): L=-4.9983, t=-3.9510, Reject
  mu2-mu3: L=1.9900, t=1.2844, Fail to reject

(vii) Power
la

### Problem 7

In [8]:
pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 3.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]
Note: you may need to restart the kernel to use updated packages.


In [18]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("HW3BlockData.csv")
df.columns = ['y', 'treatment', 'block']

# Grand mean
grand_mean = df['y'].mean()
I = df['treatment'].nunique()  # 4 treatments
J = df['block'].nunique()      # 3 blocks
K = 2                          # replicates per cell
N = len(df)

treat_means = df.groupby('treatment')['y'].mean()
block_means = df.groupby('block')['y'].mean()

SSTr = J * K * ((treat_means - grand_mean)**2).sum()
SSB  = I * K * ((block_means - grand_mean)**2).sum()
SST  = ((df['y'] - grand_mean)**2).sum()
SSE  = SST - SSTr - SSB

df_tr  = I - 1
df_b   = J - 1
df_e   = N - I - J + 1

MSTr = SSTr / df_tr
MSB  = SSB  / df_b
MSE  = SSE  / df_e

F_stat = MSTr / MSE
p_val  = 1 - stats.f.cdf(F_stat, df_tr, df_e)

print(f"{'Source':<20} {'df':>4} {'SS':>10} {'MS':>10} {'F':>8} {'p-value':>10}")
print(f"{'Treatment':<20} {df_tr:>4} {SSTr:>10.4f} {MSTr:>10.4f} {F_stat:>8.4f} {p_val:>10.4f}")
print(f"{'Block':<20} {df_b:>4} {SSB:>10.4f} {MSB:>10.4f} {'':>8} {'':>10}")
print(f"{'Error':<20} {df_e:>4} {SSE:>10.4f} {MSE:>10.4f} {'':>8} {'':>10}")
print(f"{'Total':<20} {N-1:>4} {SST:>10.4f}")

print(f"\nF({df_tr}, {df_e}) = {F_stat:.4f},  p-value = {p_val:.4f}")
if p_val < 0.05:
    print("Reject H0: Drug dosage is significantly related to y.")
else:
    print("Fail to reject H0: No significant effect of drug dosage.")

Source                 df         SS         MS        F    p-value
Treatment               3    74.8379    24.9460   6.1300     0.0046
Block                   2   161.6808    80.8404                    
Error                  18    73.2508     4.0695                    
Total                  23   309.7696

F(3, 18) = 6.1300,  p-value = 0.0046
Reject H0: Drug dosage is significantly related to y.


In [14]:
import numpy as np

# Data
data = {
    (0,0): [100.7, 102.4], (0,1): [104.3, 109.2], (0,2): [101.7,  99.4],
    (1,0): [103.5, 104.0], (1,1): [107.7, 105.9], (1,2): [104.5, 102.3],
    (2,0): [ 97.8,  94.6], (2,1): [105.2, 102.9], (2,2): [ 98.0,  99.9],
    (3,0): [102.6, 102.2], (3,1): [106.8, 106.6], (3,2): [100.1,  96.0],
}

y, X_rd, X_fr = [], [], []

for i in range(4):
    for j in range(3):
        for k in range(2):
            y.append(data[(i,j)][k])

            # Rank-deficient: (mu, a1,a2,a3,a4, b1,b2,b3)
            row = [0]*8
            row[0]=1; row[1+i]=1; row[5+j]=1
            X_rd.append(row)

            # Full-rank: (mu, a1,a2,a3, b1,b2) with sum-to-zero
            row = [0]*6
            row[0]=1
            if i < 3: row[1+i]=1
            else: row[1]=-1; row[2]=-1; row[3]=-1
            if j < 2: row[4+j]=1
            else: row[4]=-1; row[5]=-1
            X_fr.append(row)

y    = np.array(y)
X_rd = np.array(X_rd, dtype=float)
X_fr = np.array(X_fr, dtype=float)

# (i) Rank-deficient solution via pseudoinverse
beta_rd = np.linalg.pinv(X_rd) @ y
print("beta_RD:", np.round(beta_rd, 4))

# (ii) Full-rank solution
beta_fr = np.linalg.inv(X_fr.T @ X_fr) @ X_fr.T @ y
print("beta_FR:", np.round(beta_fr, 4))
a4 = -beta_fr[1]-beta_fr[2]-beta_fr[3]
b3 = -beta_fr[4]-beta_fr[5]
print(f"a4 (derived) = {a4:.4f},  b3 (derived) = {b3:.4f}")

# (iii) Contrasts
c1    = np.array([0, 1,-1, 0, 0, 0, 0, 0])
c2    = np.array([0, 1,-1/3,-1/3,-1/3, 0, 0, 0])
c3    = np.array([0, 1, 1, 0, 0,-1,-1, 0])
c1_fr = np.array([0, 1,-1, 0, 0, 0])
c2_fr = np.array([0, 4/3, 0, 0, 0, 0])
c3_fr = np.array([0, 1, 1, 0,-1,-1])

print(f"c1  @ beta_RD = {c1  @ beta_rd:.4f},  c1* @ beta_FR = {c1_fr @ beta_fr:.4f}")
print(f"c2  @ beta_RD = {c2  @ beta_rd:.4f},  c2* @ beta_FR = {c2_fr @ beta_fr:.4f}")
print(f"c3  @ beta_RD = {c3  @ beta_rd:.4f},  c3* @ beta_FR = {c3_fr @ beta_fr:.4f}")

beta_RD: [64.6921 16.6939 18.3939 13.4772 16.1272 20.1099 25.2099 19.3724]
beta_FR: [102.4292   0.5208   2.2208  -2.6958  -1.4542   3.6458]
a4 (derived) = -0.0458,  b3 (derived) = -2.1917
c1  @ beta_RD = -1.7000,  c1* @ beta_FR = -1.7000
c2  @ beta_RD = 0.6944,  c2* @ beta_FR = 0.6944
c3  @ beta_RD = -10.2320,  c3* @ beta_FR = 0.5500


### Problem 9

In [6]:
import numpy as np
from scipy import stats

# Data
group1 = [8, 10, 12, 10, 13, 12, 12, 15, 13, 9]
group2 = [8, 14, 16, 14, 10, 11, 10,  9,  9, 12]
group3 = [11, 12, 11, 23, 19, 11, 17, 17, 16, 16]
group4 = [13, 17, 20, 15, 11, 17, 16,  5, 11, 20]

groups = [group1, group2, group3, group4]
labels = ["Group1", "Group2", "Group3", "Group4"]
K = len(groups)
n_k = [len(g) for g in groups]
n   = sum(n_k)


# (i)  KRUSKAL-WALLIS TEST 

print("(i)  KRUSKAL-WALLIS TEST")

# Step 1: Combine and rank all observations (average ties)
all_data = []
for k, g in enumerate(groups):
    for obs in g:
        all_data.append((obs, k))          # (value, group_index)

all_values = np.array([x[0] for x in all_data])
ranks      = stats.rankdata(all_values)    # average rank for ties

print("\nAssign ranks (average for ties):")
print(f"{'Obs':>6}  {'Group':>7}  {'Rank':>6}")
for (val, grp), r in zip(all_data, ranks):
    print(f"{val:>6}  {labels[grp]:>7}  {r:>6.2f}")

# Step 2: Rank sums per group
rank_sums = np.zeros(K)
for i, (val, grp) in enumerate(all_data):
    rank_sums[grp] += ranks[i]

print("\nRank sums per group:")
for k in range(K):
    print(f"  R_{k+1} = {rank_sums[k]:.2f}  (n_{k+1} = {n_k[k]})")

# Step 3: Tie correction factor
unique_vals, counts = np.unique(all_values, return_counts=True)
tie_groups = counts[counts > 1]
C = 1 - np.sum(tie_groups**3 - tie_groups) / (n**3 - n)
print(f"\nTie correction factor C = {C:.6f}")
print(f"  Tied values and their counts:")
for v, c in zip(unique_vals, counts):
    if c > 1:
        print(f"    value={v}, count={c}, t^3-t={c**3-c}")

# Step 4: Compute H statistic
H_raw = (12 / (n * (n + 1))) * np.sum(rank_sums**2 / np.array(n_k)) - 3 * (n + 1)
H     = H_raw / C

print(f"\nH statistic:")
print(f"  H (uncorrected) = {H_raw:.4f}")
print(f"  H (tie-corrected) = H / C = {H:.4f}")

# Step 5: Critical value and decision
alpha   = 0.05
df      = K - 1
chi2_cv = stats.chi2.ppf(1 - alpha, df)
p_value = 1 - stats.chi2.cdf(H, df)

print(f"\nDecision (chi-distribution, df = {df})")
print(f"  Critical value chi_{{0.05, {df}}} = {chi2_cv:.4f}")
print(f"  H = {H:.4f},  p-value = {p_value:.4f}")
if H > chi2_cv:
    print("  Reject H0: group medians are not all equal.")
else:
    print("  Fail to reject H0.")

# Verify with scipy
H_scipy, p_scipy = stats.kruskal(*groups)
print(f"\n  [scipy verify]  H = {H_scipy:.4f},  p = {p_scipy:.4f}")


# (ii)  Permutation Test

print("(ii)  Permutation Test")

rng = np.random.default_rng(seed=42)
B   = 1000          # number of permutations

# Observed test statistic: F-statistic (one-way ANOVA)
def f_statistic(groups):
    k     = len(groups)
    ns    = [len(g) for g in groups]
    N     = sum(ns)
    means = [np.mean(g) for g in groups]
    grand = np.mean(np.concatenate(groups))
    SSB   = sum(ns[i] * (means[i] - grand)**2 for i in range(k))
    SSW   = sum(np.sum((np.array(g) - means[i])**2)
                for i, g in enumerate(groups))
    MSB   = SSB / (k - 1)
    MSW   = SSW / (N - k)
    return MSB / MSW

all_obs   = np.concatenate(groups)
F_obs     = f_statistic(groups)
print(f"observed F-statistic = {F_obs:.4f}")

# Permutation loop
F_perm = np.empty(B)
for b in range(B):
    perm      = rng.permutation(all_obs)
    perm_grps = [perm[sum(n_k[:i]):sum(n_k[:i+1])] for i in range(K)]
    F_perm[b] = f_statistic(perm_grps)

p_perm = np.mean(F_perm >= F_obs)

print(f"Number of permutations B = {B}")
print(f"Permutation F-value = {F_obs:.4f}")
print(f"Permutation p-value = {p_perm:.4f}")


if p_perm < alpha:
    print("  Reject H0: group medians are not all equal.")
else:
    print("  Fail to reject H0.")

# Summary
print("\n" + "=" * 65)
print("Summary")
print("=" * 65)
print(f"  Kruskal-Wallis:  H = {H:.4f},  p = {p_value:.4f}")
print(f"  Permutation:     F = {F_obs:.4f},  p = {p_perm:.4f}")
print(f"  Both tests at α = {alpha}")


(i)  KRUSKAL-WALLIS TEST

Assign ranks (average for ties):
   Obs    Group    Rank
     8   Group1    2.50
    10   Group1    8.50
    12   Group1   19.00
    10   Group1    8.50
    13   Group1   23.00
    12   Group1   19.00
    12   Group1   19.00
    15   Group1   27.50
    13   Group1   23.00
     9   Group1    5.00
     8   Group2    2.50
    14   Group2   25.50
    16   Group2   30.50
    14   Group2   25.50
    10   Group2    8.50
    11   Group2   13.50
    10   Group2    8.50
     9   Group2    5.00
     9   Group2    5.00
    12   Group2   19.00
    11   Group3   13.50
    12   Group3   19.00
    11   Group3   13.50
    23   Group3   40.00
    19   Group3   37.00
    11   Group3   13.50
    17   Group3   34.50
    17   Group3   34.50
    16   Group3   30.50
    16   Group3   30.50
    13   Group4   23.00
    17   Group4   34.50
    20   Group4   38.50
    15   Group4   27.50
    11   Group4   13.50
    17   Group4   34.50
    16   Group4   30.50
     5   Group4    1.00
    1